# ComfyUI icon setup / Flux.1 Dev

Kick×Kickアイコン生成用。

## 重要
- OllamaGeminiは使わない。
- BRIA_RMBG / ConvertRasterToVector / SaveSVG は使わない。
- 現在の動作対象はPNGアイコン生成まで。
- SVG化・背景透過は、採用ノードを決めてから別途追加する。


## Cell 1: 基本設定

In [ ]:
import os, shutil, zipfile, glob
from pathlib import Path

GPU_TIER = os.environ.get('GPU_TIER', '16GB')
WORK_DIR = '/workspace/runpod-slim'
COMFY_DIR = f'{WORK_DIR}/ComfyUI'
MODEL_DIR = '/comfyui_models'
WORKFLOWS_DIR = f'{COMFY_DIR}/user/default/workflows'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(WORKFLOWS_DIR, exist_ok=True)

zip_candidates = sorted(glob.glob(f'{WORK_DIR}/comfyui_backup_*.zip'))
if zip_candidates:
    zip_path = zip_candidates[-1]
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORK_DIR)
    os.rename(zip_path, zip_path + '.extracted')
    print('backup zip extracted')
else:
    print('no backup zip')

print('GPU_TIER:', GPU_TIER)
print('WORK_DIR:', WORK_DIR)
print('COMFY_DIR:', COMFY_DIR)
print('MODEL_DIR:', MODEL_DIR)


## Cell 2: extra_model_paths.yaml生成

In [ ]:
from pathlib import Path
yaml_path = Path(f'{COMFY_DIR}/extra_model_paths.yaml')
yaml_path.parent.mkdir(parents=True, exist_ok=True)
yaml_path.write_text(f'''comfyui:
    base_path: {MODEL_DIR}/
    is_default: true
    checkpoints: checkpoints/
    clip: text_encoders/
    text_encoders: text_encoders/
    diffusion_models: diffusion_models/
    unet: diffusion_models/
    vae: vae/
    loras: loras/
    upscale_models: upscale_models/
''')
print('extra_model_paths.yaml written:', yaml_path)


## Cell 3: iconファイル配置

In [ ]:
import glob, shutil, os
workflow_candidates = glob.glob(f'{WORK_DIR}/flux1_dev_icon_*_workflow_v2ollama.json')
for src in workflow_candidates:
    dst = f'{WORKFLOWS_DIR}/{os.path.basename(src)}'
    shutil.copy(src, dst)
    print('workflow copied:', dst)

html_src = f'{WORK_DIR}/comfyui_icon_mobile.html'
print('html exists:', os.path.exists(html_src), html_src)
print('icon setup policy: PNG only / no OllamaGemini / no SVG nodes')
